# Inline Citations with LlamaCloud with Streaming
In this notebook we show you how to perform inline citations with LlamaCloud. 

## Setup

Install core packages, download files. You will need to upload these documents to LlamaCloud.

In [ ]:
!pip install llama-index
!pip install llama-index-core
!pip install llama-index-embeddings-openai
!pip install llama-index-question-gen-openai
!pip install llama-index-postprocessor-flag-embedding-reranker
!pip install git+https://github.com/FlagOpen/FlagEmbedding.git
!pip install llama-parse

In [ ]:
# download Apple 
!wget "https://s2.q4cdn.com/470004039/files/doc_earnings/2023/q4/filing/_10-K-Q4-2023-As-Filed.pdf" -O data/apple_2023.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2022/q4/_10-K-2022-(As-Filed).pdf" -O data/apple_2022.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2021/q4/_10-K-2021-(As-Filed).pdf" -O data/apple_2021.pdf
!wget "https://s2.q4cdn.com/470004039/files/doc_financials/2020/ar/_10-K-2020-(As-Filed).pdf" -O data/apple_2020.pdf
!wget "https://www.dropbox.com/scl/fi/i6vk884ggtq382mu3whfz/apple_2019_10k.pdf?rlkey=eudxh3muxh7kop43ov4bgaj5i&dl=1" -O data/apple_2019.pdf

# download Tesla
!wget "https://ir.tesla.com/_flysystem/s3/sec/000162828024002390/tsla-20231231-gen.pdf" -O data/tesla_2023.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000095017023001409/tsla-20221231-gen.pdf" -O data/tesla_2022.pdf
!wget "https://www.dropbox.com/scl/fi/ptk83fmye7lqr7pz9r6dm/tesla_2021_10k.pdf?rlkey=24kxixeajbw9nru1sd6tg3bye&dl=1" -O data/tesla_2021.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000156459021004599/tsla-10k_20201231-gen.pdf" -O data/tesla_2020.pdf
!wget "https://ir.tesla.com/_flysystem/s3/sec/000156459020004475/tsla-10k_20191231-gen_0.pdf" -O data/tesla_2019.pdf

Some OpenAI and LlamaParse details. The OpenAI LLM is used for response synthesis.

In [1]:
# llama-parse is async-first, running the async code in a notebook requires the use of nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
# API access to llama-cloud
os.environ["LLAMA_CLOUD_API_KEY"] = ""

In [3]:
# Using OpenAI API for embeddings/llms
os.environ["OPENAI_API_KEY"] = ""

## Load Documents into LlamaCloud

The first order of business is to download the 5 Apple and Tesla 10Ks and upload them into LlamaCloud.

You can easily do this by creating a pipeline and uploading docs via the "Files" mode.

After this is done, proceed to the next section.

## Define NodeCitationPostProcessor
Add node id to metadata to match the citation links

In [4]:
from typing import List, Optional

from llama_index.core import QueryBundle
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore


class NodeCitationProcessor(BaseNodePostprocessor):
    """
    Append node_id into metadata for citation purpose.
    Config SYSTEM_CITATION_PROMPT in your runtime environment variable to enable this feature.
    """

    def _postprocess_nodes(
        self,
        nodes: List[NodeWithScore],
        query_bundle: Optional[QueryBundle] = None,
    ) -> List[NodeWithScore]:
        for node_score in nodes:
            node_score.node.metadata["node_id"] = node_score.node.node_id
        return nodes

## Define System Citation Prompt
Modify the system prompt to add the citation links based on the metadata

In [5]:
SYSTEM_CITATION_PROMPT = """You have provided information from a knowledge base that has been passed to you in nodes of information.
Each node has useful metadata such as node ID, file name, page, etc.
Please add the citation to the data node for each sentence or paragraph that you reference in the provided information.
The citation format is: . [citation:<node_id>]()
Where the <node_id> is the unique identifier of the data node.

Example:
We have two nodes:
  node_id: xyz
  file_name: llama.pdf
  
  node_id: abc
  file_name: animal.pdf

User question: Tell me a fun fact about Llama.
Your answer:
A baby llama is called "Cria" [citation:xyz]().
It often live in desert [citation:abc]().
It\\'s cute animal."""

## Define LlamaCloud Retriever over Documents

In this section we define LlamaCloud Retriever over these documents.

In [8]:
from llama_index.indices.managed.llama_cloud import LlamaCloudIndex
import os

index = LlamaCloudIndex(
  name="apple_demo",
  project_name="llamacloud_demo",
  api_key=os.environ["LLAMA_CLOUD_API_KEY"]
)

#### Define chunk retriever

The chunk-level retriever does vector search with a final reranked set of `rerank_top_n=5`.

In [16]:
chunk_retriever = index.as_retriever(
    retrieval_mode="chunks",
    rerank_top_n=5
)
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini", system_prompt=SYSTEM_CITATION_PROMPT)
query_engine = RetrieverQueryEngine.from_args(
    chunk_retriever, 
    llm=llm,
    response_mode="tree_summarize",
    node_postprocessors=[NodeCitationProcessor()],
    streaming=True
)

## Generate final output matching citations with page labela
Given the found nodes, match the page assigned and build a final url

In [85]:
import re
from typing import Generator

# Accepts [citation:ID] and [citation:ID]() (spaces/newlines allowed, case-insensitive)
_CITATION_RX   = re.compile(r'\[\s*citation\s*:\s*([^\]]+?)\s*\]\s*(?:\(\s*\))?', re.IGNORECASE)
# Detects a trailing, incomplete "[citation:" at the END of a string
_INCOMPLETE_RX = re.compile(r'\[\s*citation\s*:\s*[^\]]*$', re.IGNORECASE)

def stream_citations_with_sources(resp, check_every: int = 64) -> Generator[str, None, None]:
    """
    Incrementally replace [citation:ID] / [citation:ID]() with:
      [n](https://fake.url/SampleFile#page=<page_label>)
    Emits only the *new* safe prefix each time; never flushes partial tags.
    """

    # Build id -> page_label now (OK if empty; we'll use 'unknown')
    nodes = getattr(resp, "source_nodes", []) or []
    id_to_label = {str(n.id_): n.metadata.get("page_label", "unknown") for n in nodes}

    order: dict[str, int] = {}
    counter = [1]  # mutable to avoid nonlocal

    def _link_for(cid: str) -> str:
        cid = cid.strip()
        if cid not in order:
            order[cid] = counter[0]
            counter[0] += 1
        n = order[cid]
        page = id_to_label.get(cid, "unknown")
        # TODO: replace fake URL with node.metadata["web_url"] when available
        return f"[{n}](https://fake.url/SampleFile#page={page})"

    def _replace_complete(text: str) -> str:
        def _repl(m: re.Match) -> str:
            return _link_for(m.group(1))
        # Replace only complete tags; do NOT strip any incomplete tail here
        return _CITATION_RX.sub(_repl, text)

    acc = ""           # full accumulated text so far
    emitted_upto = 0   # index in acc we've already emitted
    since = 0

    for chunk in resp.response_gen:
        acc += chunk
        acc = _replace_complete(acc)          # replace anywhere tags became complete
        since += len(chunk)

        # Find safe end: don't include a trailing incomplete "[citation:"
        safe_end = len(acc)
        m = _INCOMPLETE_RX.search(acc)
        if m and m.end() == len(acc):
            safe_end = m.start()

        # Emit only the newly available safe prefix
        if safe_end > emitted_upto and (']' in chunk or ')' in chunk or since >= check_every):
            yield acc[emitted_upto:safe_end]
            emitted_upto = safe_end
            since = 0

    # End of stream: drop any dangling start, replace once more, emit the rest
    tail = acc[emitted_upto:]
    if tail:
        # Remove trailing incomplete start if present
        m = _INCOMPLETE_RX.search(tail)
        if m and m.end() == len(tail):
            tail = tail[:m.start()]
        tail = _replace_complete(tail)
        if tail:
            yield tail

## Query it

In [86]:
query = "What are the tiny risks for apple"

In [92]:
from IPython.display import display, HTML

resp = query_engine.query(query)

buf = []
handle = display(HTML("<pre></pre>"), display_id=True)

for part in stream_citations_with_sources(resp):
    buf.append(part)
    html = "<pre>" + "".join(buf) + "</pre>"
    handle.update(HTML(html))